# Model comparison with cross-validation, then the threshold

One row is one telco customer; `Churn` is `Yes` when the customer left in the last month. The retention manager: "Each customer you flag gets a retention offer, which costs us 100 dollars whether they would have left or not. A customer who leaves without an offer costs us 400 dollars. Which model, and how many customers should we flag?"

## The starter

The cell repeats D01 5.6 and D02: `TotalCharges` converted to numbers, the `ColumnTransformer` `pre` (median and scaling for numbers, one-hot for text), a second one `tree_pre` with no scaling for the tree models, and the 80 / 20 split with `stratify=y, random_state=42`. Your 5,634 training customers and 1,409 test customers are the same as yesterday's.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
tree_pre = ColumnTransformer([          # the same blocks without the scaler: trees do not need it
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], verbose_feature_names_out=False).set_output(transform="pandas")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(len(X_train), "training customers,", len(X_test), "test customers")

## Step 1 · One set of folds for every model

`cv = StratifiedKFold(5, shuffle=True, random_state=42)`, created once and reused.

## Step 2 · Cross-validate four models

For each model, run `cross_validate(model, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision", "neg_brier_score"], return_train_score=True)`. Every model sits inside its own pipeline, so every fold fits its own preprocessing.

- `make_pipeline(pre, DummyClassifier(strategy="prior"))`
- `make_pipeline(pre, LogisticRegression(max_iter=1000))`
- `make_pipeline(tree_pre, RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))`
- `make_pipeline(tree_pre, HistGradientBoostingClassifier(random_state=42))`

## Step 3 · One table

Mean and standard deviation of the validation ROC AUC, mean average precision, mean Brier score (minus `test_neg_brier_score`), and the mean training ROC AUC.

## Step 4 · "Which model ranks customers best?"

Translate: the highest mean validation ROC AUC. Is the gap between the top two models larger than one standard deviation? Which model has the largest gap between training and validation AUC, and what does that gap mean (D02 3.1)?

## Step 5 · The cost as a function

`cost = 100 * FP + 400 * FN + 100 * TP`, from `confusion_matrix(y_true, y_pred).ravel()`. Wrap minus the mean cost per customer in `make_scorer`: the tuner maximizes its score, so a lower cost must be a higher score.

## Step 6 · Choose the threshold on the folds

`TunedThresholdClassifierCV(model, scoring=scorer, cv=cv, random_state=42).fit(X_train, y_train)`. Print `best_threshold_` and minus `best_score_` (the cost per customer on the folds).

## Step 7 · Only now, the test customers, once

The total cost at the 0.5 default (`model.predict`) and at your threshold (`tuned.predict`), and how many customers each flags.

## Step 8 · "How many customers should we flag, and what does that save?"

Answer in two sentences with the numbers.

## If you chose the boosted model

## Check yourself